In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

In [2]:
import numpy as np
import random
from tqdm import tqdm
import os
from collections import Counter
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [3]:
class Config:
    SEED=42
    MAX_VOCAB_SIZE = 25000
    MAX_LEAN = 256
    BATCH_SIZE = 64
    EMBEDDING_DIM = 128
    HIDDEN_DIM = 256
    NUM_LAYERS = 2
    DROPOUT = 0.5
    LR=0.01
    WEIGHT_DECAY=1e-5
    EPOCHS=20
    GRAD_CLIP=1.0
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_PATH='best_RNN_sentiment_model.pth'

# Set seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)

In [ ]:
# Data Pre-Processing
def clean_text(text):
    text = re.sub(r'<.*>','',text)
    text = re.sub(r'[^a-zA-Z\s]', '',text)
    text = text.lower().strip()
    return text

In [5]:
class IMDBDataset(Dataset):
    def __init__(self,texts,labels,vocab=None,max_len=256):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        tokens = [self.vocab.get(word,self.vocab['<unk>']) for word in text.split()][:self.max_len]
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [6]:
def build_vocab(texts,max_size=25000):
    word_freq = Counter()
    for text in texts:
        word_freq.update(text.split())
    
    vocab = {'<pad>':0 , '<unk>':1}
    for word, freq in word_freq.most_common(max_size-2):
        vocab[word] = len(vocab)

    print(f'Vocabulary size: {len(vocab)}')
    return vocab

In [7]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return texts_padded, labels

In [8]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout=0.5):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers>1 else 0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim,1)

    def forward(self,x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        output, hidden = self.rnn(embedded)

        # Take the last layers's last hidden state
        hidden = hidden[-1]
        hidden = self.dropout(hidden)

        out = self.fc(hidden)
        return out.squeeze(1)

In [ ]:
def train_epoch(model, dataloader,optimizer,criterion,device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for texts, labels in tqdm(dataloader, desc='Training'):
        texts, labels = texts.to(device), labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.paramters(), Config.GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()

        preds = (torch.sigmoid(outputs)>0.5).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    